# PARFLM on TinyStories — P10 sparsity ladder (k=8, k=16) on A100 / H100

## Context

The P10 ladder (P10a–P10h) established the PARFLM architectural ceiling at **val PPL ≈ 26.4** on TinyStories using the full P5+P7+P8 stack with top-k=4 Gumbel-softmax sparsity. All P10 cells used k=4 — the winner of the small-scale P5 Shakespeare sparsity ladder.

This notebook runs the **k=8 and k=16 points** of the sparsity ladder at TinyStories / H100 scale, using the P10g architecture (the configuration that first reached the ceiling). The purpose is to determine whether k=4 remains the optimal sparsity point at the flagship TinyStories scale, or whether the richer context (T=512 vs T=128, 5M vs 321K tokens) changes the optimal operating point.

### Small-scale reference (P5 Shakespeare ladder, d=128, T=128)

| Sparse cell | Val PPL  | Final γ  |
| --- | ---: | ---: |
| **k=4 (P5 winner)** | **176.65** | 0.134 |
| k=8 | 218.73 | 0.134 |
| k=16 | 205.25 | 0.085 |
| k=32 | NaN (failed) | — |

### TinyStories baseline (P10g, d=256, L=8, T=512, k=4)

| Cell | Steps | Best val PPL | Final val PPL |
| --- | --- | --- | --- |
| P10g (k=4, 16k steps) | 16000 | **26.42** | 27.16 |

### This notebook's cells

| Cell | Composition | Predicted val PPL | What it tests |
| --- | --- | --- | --- |
| `P10i` | P10g architecture + **k=8** | 28–35 | Does doubling k degrade PPL at TinyStories scale? At Shakespeare scale k=8 was 24% worse than k=4. |
| `P10j` | P10g architecture + **k=16** | 27–33 | Does quadrupling k degrade PPL? At Shakespeare scale k=16 was 16% worse than k=4 (non-monotone ordering). |

### Pre-registered predictions

1. **k=4 remains optimal**: the score head's ability to learn good top-k rankings degrades with k (Gumbel signal dilution), and TinyStories scale does not overcome this — PPL ordering remains k=4 < k=16 < k=8 (same non-monotone pattern as Shakespeare).
2. **Gap narrows at scale**: the absolute PPL gap between k=4 and k=8/k=16 is smaller at TinyStories scale than at Shakespeare scale (the 5M-token corpus provides richer gradient signal for the score head).
3. **γ suppression at k=16**: the integrator re-suppresses γ to compensate for noisier routing, as observed at Shakespeare scale (k=16 γ=0.085 vs k=4 γ=0.134).

### Decision rule

- If both k=8 and k=16 land above P10g (26.42) → **k=4 confirmed as optimum at TinyStories scale**. Report the three-point sparsity ladder in paper v4.
- If either k=8 or k=16 beats P10g → **surprising result**; re-run the winner at n=3 seeds for confirmation, then update the paper narrative.
- If k=32 is attempted and NaN-diverges again → **consistent with Shakespeare-scale failure mode** (Gumbel gradient overshoot); omit from the main ladder table, cite the forensic analysis from P5.

## 0. Environment setup + cell selector

Pick which cell to run via the `CELL` constant. Architecture is locked to P10g (the configuration that reached the 26.4 ceiling at k=4) — only `top_k` varies.

**Colab bootstrap (automatic).** When run on Google Colab the next cell:

1. mounts your Google Drive at `/content/drive`,
2. shallow-clones (`--depth 1 --branch main`) the public `dimitarpg13/semsimula` repo into `/content/semsimula` (or refreshes it if already present),
3. symlinks the data cache (`notebooks/conservative_arch/data`) to `/content/drive/MyDrive/semsimula_parflm/data` so the TinyStories tokenisation only happens once, ever,
4. routes **all** training outputs (ckpt, training log, val PPL plot, P6 diagnostic, s_ℓ profile) to `/content/drive/MyDrive/semsimula_parflm/p10_tinystories/{cell}/seed{SEED}/` — i.e. nothing important is written into the ephemeral `/content/semsimula` clone.

When run locally (off Colab) the notebook walks up from the CWD to find the repo root and writes to the in-repo path `parf/results/p10_tinystories/{cell}/seed{SEED}/` as before.

In [ ]:
# ===== Pick the cell to run =====
CELL = 'P10i'   # one of: 'P10i' (k=8) | 'P10j' (k=16)
SEED = 0

# ===== Source-of-truth repo + GDrive output dir =====
REPO_URL          = 'https://github.com/dimitarpg13/semsimula.git'
REPO_BRANCH       = 'main'
COLAB_REPO_PATH   = '/content/semsimula'                     # ephemeral source
GDRIVE_OUT_REL    = 'semsimula_parflm'                        # under /content/drive/MyDrive/

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    """Run a shell command, stream output, raise on non-zero exit."""
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    # ---------- (a) Mount Google Drive ----------
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    # ---------- (b) Shallow-clone (or refresh) the semsimula repo ----------
    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    # ---------- (c) Pin tokenisation cache to Drive (persists across sessions) ----------
    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.exists():
        try:
            repo_data_dir.rmdir()
        except OSError:
            print(f'NOTE: {repo_data_dir} is non-empty; leaving as-is (no Drive symlink).')
    if not repo_data_dir.exists():
        repo_data_dir.symlink_to(DATA_CACHE, target_is_directory=True)
        print(f'data cache symlink: {repo_data_dir} -> {DATA_CACHE}')

    # ---------- (d) Output root: ckpt / log / plots / diagnostic all live on Drive ----------
    RESULTS_ROOT = GDRIVE_OUT / 'p10_tinystories'
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    # ---------- (e) Make sure the data_module's deps are present ----------
    _sh('pip install -q transformers huggingface_hub pyarrow')

else:
    # ---------- Local / non-Colab: locate repo root by walking up from CWD ----------
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError(
            'Could not locate the semsimula repo root from the notebook CWD. '
            'cd into the repo before launching the notebook.'
        )
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
        / 'results' / 'p10_tinystories'
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ===== sys.path so we can import data_module / model_parf / scaleup utils =====
PARF_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
SCALEUP_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
DATA_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch'
for p in (str(REPO_ROOT), str(DATA_DIR), str(SCALEUP_DIR), str(PARF_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'\nREPO_ROOT     = {REPO_ROOT}')
print(f'PARF_DIR      = {PARF_DIR}')
print(f'SCALEUP_DIR   = {SCALEUP_DIR}')
print(f'RESULTS_ROOT  = {RESULTS_ROOT}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## 1. Disable TF32, set seeds, pick device

In [ ]:
import torch
import numpy as np

# ----- TF32 OFF on Ampere/Hopper -----
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    print(f'CUDA: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    if cap[0] < 8:
        print(f'  WARNING: compute capability < 8.0 ({cap}); designed '
              f'for A100 (sm_80) or H100 (sm_90).')
    print(f'  TF32 matmul = {torch.backends.cuda.matmul.allow_tf32}  (must be False)')
    print(f'  TF32 cuDNN  = {torch.backends.cudnn.allow_tf32}  (must be False)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device — TF32 toggles are CUDA-only.')
else:
    device = 'cpu'
    print('CPU only — P10 cells take ~days on CPU; smoke-test mode strongly recommended.')
print(f'\ndevice = {device}')

## 2. Load TinyStories + the bundled logfreq surprisal

P10h showed that 20M tokens yields zero improvement over 5M at 16k steps. All sparsity-ladder cells use 5M tokens (1 shard).

In [ ]:
from data_module import load_tiny_stories, get_batch

MAX_TRAIN_TOKENS = 5_000_000
N_TRAIN_FILES = 1
train_ids, val_ids = load_tiny_stories(
    n_train_files=N_TRAIN_FILES, max_train_tokens=MAX_TRAIN_TOKENS
)
print(f'tokens: train={len(train_ids):,}  val={len(val_ids):,}  '
      f'(train capped at {MAX_TRAIN_TOKENS:,})')
print(f'train_ids dtype={train_ids.dtype}  range=[{train_ids.min()}, {train_ids.max()}]')

BUNDLED_LOGFREQ = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ   = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'
if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
    print(f'Using bundled TinyStories logfreq surprisal: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached TinyStories logfreq surprisal: {LOGFREQ_PATH}')
else:
    VOCAB_SIZE = 50257
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

## 3. Translate `CELL` switch into trainer flags + build config

Both cells use the P10g architecture (full P5+P7+P8 stack, v_hidden=2048, v_phi inner 64, 16k steps). The **only** difference is `top_k`.

In [ ]:
from train_parf_scaleup import build_config
from model_parf import PARFLM
from model_parf_sparse import SparsePARFLM, SparsePARFConfig
import torch.nn.functional as F

SPARSITY_RECIPES = {
    # ---- P10i: k=8 at TinyStories scale ----
    # P10g architecture (full P5+P7+P8, v_hidden=2048, vphi=64, 16k steps)
    # with top-k doubled from 4 to 8.  At Shakespeare scale, k=8 landed at
    # 218.73 PPL — 24% worse than k=4 (176.65).  The score head at k=8
    # suffered from Gumbel signal dilution: routing noise doubled, and the
    # per-candidate gradient shrank by ~2x.  At TinyStories scale (T=512,
    # 5M tokens) the richer corpus and longer context may partially
    # compensate, but the prediction is k=8 still loses to k=4.
    # Pre-registered predictions:
    #   * val PPL: 28–35 (worse than P10g's 26.42)
    #   * γ: ~0.10–0.14 (similar to or slightly below P10g's 0.134)
    #   * If PPL > 35: score head is severely capacity-limited at k=8;
    #     the Shakespeare-scale degradation pattern transfers to scale-up.
    #   * If PPL < 27: surprising — k=8 is competitive at TinyStories
    #     scale despite Shakespeare evidence to the contrary.
    'P10i': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=64, v_phi_theta_hidden=64,
        v_hidden=2048, v_depth=None,
        _top_k=8,
        _steps_override=16000,
    ),
    # ---- P10j: k=16 at TinyStories scale ----
    # Same as P10i but top-k=16.  At Shakespeare scale, k=16 landed at
    # 205.25 PPL — 16% worse than k=4, but better than k=8 (non-monotone).
    # The k=16 cell showed γ suppression (0.085 vs 0.134), suggesting the
    # integrator compensated for noisier routing by reducing dissipation.
    # Pre-registered predictions:
    #   * val PPL: 27–33 (worse than P10g, possibly better than P10i)
    #   * γ: ~0.07–0.10 (γ suppression, as observed at Shakespeare scale)
    #   * Non-monotone ordering k=4 < k=16 < k=8 may persist.
    #   * If PPL < 26.5: the richer context at TinyStories lets k=16's
    #     broader attention field compensate for routing noise — would
    #     overturn the small-scale story.
    'P10j': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=64, v_phi_theta_hidden=64,
        v_hidden=2048, v_depth=None,
        _top_k=16,
        _steps_override=16000,
    ),
}
if CELL not in SPARSITY_RECIPES:
    raise ValueError(f'CELL must be one of {list(SPARSITY_RECIPES)}; got {CELL!r}')
recipe = dict(SPARSITY_RECIPES[CELL])  # shallow copy

# Extract notebook-level overrides (prefixed with _) before passing to build_config.
_STEPS_OVERRIDE = recipe.pop('_steps_override', None)
_TOP_K = recipe.pop('_top_k', 4)

print(f'P10 sparsity-ladder recipe for {CELL}:')
print(f'  {"top_k":28s} = {_TOP_K}')
for k, v in recipe.items():
    print(f'  {k:28s} = {v!r}')
if _STEPS_OVERRIDE is not None:
    print(f'  {"_steps_override":28s} = {_STEPS_OVERRIDE}')

_BC_OVERRIDES = ('v_phi_phi_hidden', 'v_phi_theta_hidden', 'v_hidden', 'v_depth')
cfg, train_cfg, tag = build_config(
    mode='scaleup',
    logfreq_path=str(LOGFREQ_PATH),
    fixed_gamma=None,
    sparse_top_k=_TOP_K,
    sparse_score_head_hidden=32,
    sparse_gumbel_tau_init=1.0,
    sparse_gumbel_tau_min=0.1,
    sparse_gumbel_noise=True,
    **{k: recipe.get(k) for k in _BC_OVERRIDES},
    **{k: v for k, v in recipe.items() if k not in _BC_OVERRIDES},
)

if _STEPS_OVERRIDE is not None:
    train_cfg['steps'] = _STEPS_OVERRIDE
    train_cfg['warmup_steps'] = min(train_cfg['warmup_steps'],
                                    _STEPS_OVERRIDE // 20)
full_tag = f'{tag}_seed{SEED}'
is_sparse = isinstance(cfg, SparsePARFConfig)
print(f'\ntag = {full_tag}')
print(f'cell shape: d={cfg.d}, L={cfg.L}, T={cfg.max_len}, '
      f'v_hidden={cfg.v_hidden}, v_phi_phi_hidden={cfg.v_phi_phi_hidden}, '
      f'v_phi_theta_hidden={cfg.v_phi_theta_hidden}')
print(f'train_cfg = {train_cfg}')
print(f'sparse: {is_sparse}, top_k={getattr(cfg, "top_k", None)}')

In [ ]:
torch.manual_seed(SEED)
Model = SparsePARFLM if is_sparse else PARFLM
model = Model(cfg).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_phi = sum(p.numel() for p in model.V_phi.parameters())
n_pls = model.raw_v_phi_scale.numel() if model.raw_v_phi_scale is not None else 0
n_score = (sum(p.numel() for p in model.score_head.parameters())
           if is_sparse else 0)
print(f'params: total={n_total:,}  V_theta={n_v_theta:,}  '
      f'V_phi={n_v_phi:,}  per_layer_scale={n_pls}  score_head={n_score:,}')
if model.raw_v_phi_scale is not None:
    print(f'init s_ℓ = {F.softplus(model.raw_v_phi_scale).detach().tolist()}')

## 4. Causal-violation probe (gates training)

In [ ]:
from causal_probe_parf import assert_causal
assert_causal(model, vocab_size=cfg.vocab_size, T=32, seed=SEED)
print('causal probe OK — no future-position leak.')

## 5. Train

16 000 steps (same as P10g), batch=16, T=512, AdamW(0.9, 0.95), cosine LR with 400-step warmup. Wall-clock estimate: H100-80GB ~6–9 h per cell.

If you want to test the loop end-to-end first without burning the full budget, override `STEPS = 200, EVAL_INTERVAL = 100` in the cell below before running.

In [ ]:
import math, time, json

BATCH = train_cfg['batch_size']
BLOCK = train_cfg['block_size']
STEPS = train_cfg['steps']
LR = train_cfg['lr']
WD = train_cfg['weight_decay']
WARMUP = train_cfg['warmup_steps']
GRAD_CLIP = train_cfg['grad_clip']
EVAL_INTERVAL = train_cfg['eval_interval']
EVAL_ITERS = train_cfg['eval_iters']
LOG_INTERVAL = train_cfg['log_interval']

from train_parf_scaleup import tau_schedule
GUMBEL_ANNEAL_FRAC = 0.8

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

opt = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()

log = []
t0 = time.time()
for step in range(STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    if is_sparse:
        model.set_gumbel_tau(tau_schedule(step, cfg.gumbel_tau_init,
                                          cfg.gumbel_tau_min, STEPS,
                                          GUMBEL_ANNEAL_FRAC))
    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        tau_str = (f'  tau={model.gumbel_tau:.3f}' if is_sparse else '')
        msg = (f'[{CELL} k={_TOP_K}] step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'train_loss={loss.item():.4f}  '
               f'gamma={model.gamma.item():.3f}{tau_str}  '
               f'wall={time.time() - t0:.0f}s')
        print(msg)
    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}')
        log.append({'step': step + 1, 'val_loss': val_loss,
                    'val_ppl': val_ppl, 'train_loss': loss.item()})

print(f'\n[{CELL} k={_TOP_K}] Training done.  total wall = {time.time() - t0:.0f}s  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}')

## 6. Save checkpoint and training log

In [ ]:
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'

import dataclasses

best_row = min(log, key=lambda r: r['val_ppl'])

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'model_cfg': dataclasses.asdict(cfg),
        'variant': full_tag,
        'cell': CELL,
        'step': STEPS,
        'top_k': _TOP_K,
        'final_val_ppl': log[-1]['val_ppl'],
        'best_val_ppl': best_row['val_ppl'],
        'best_step': best_row['step'],
        'final_gamma': float(model.gamma.item()),
        'final_gumbel_tau': model.gumbel_tau if is_sparse else None,
        'elapsed_sec': time.time() - t0,
    },
    ckpt_path,
)
with open(log_path, 'w') as f:
    for row in log:
        f.write(json.dumps(row) + '\n')
print(f'wrote ckpt: {ckpt_path}')
print(f'wrote log : {log_path}')
print(f'\nbest val_ppl = {best_row["val_ppl"]:.2f} @ step {best_row["step"]}')
print(f'final val_ppl = {log[-1]["val_ppl"]:.2f}')

## 7. Plot val PPL trajectory + bookend baselines

Reference lines: P10g k=4 ceiling (26.42), SPLM em_ln wall (33.55), MatchedGPT (7.81).

In [ ]:
import matplotlib.pyplot as plt

steps_log = [r['step'] for r in log]
ppl_log = [r['val_ppl'] for r in log]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(steps_log, ppl_log, marker='o', color='#3a6ea5', label=f'{CELL} (k={_TOP_K})')
ax.axhline(26.42, color='#d4820c', linestyle='-.',  label='P10g k=4 ceiling (26.42)')
ax.axhline(33.55, color='#888',    linestyle='--',  label='SPLM em_ln (33.55, the wall)')
ax.axhline( 7.81, color='#b03030', linestyle=':',   label='MatchedGPT (7.81)')
ax.set_xlabel('train step')
ax.set_ylabel('val PPL (log scale)')
ax.set_yscale('log')
ax.set_title(f'{CELL} (k={_TOP_K}) on TinyStories — val PPL trajectory  '
             f'(best = {min(ppl_log):.2f}, final = {ppl_log[-1]:.2f})')
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_val_ppl.png', dpi=120)
plt.show()

## 8. (Optional) Run the V_φ channel diagnostic on the trained checkpoint

Produces per-layer Φ_φ / Θ_φ / distance / V_φ histograms + gradient-ratio R(ℓ) + failure-mode read-off.  Useful for diagnosing *how* the score head's routing quality differs at k=8/k=16 vs the k=4 reference.

In [ ]:
RUN_DIAGNOSTIC = True   # set False to skip
if RUN_DIAGNOSTIC:
    import subprocess
    DIAG_OUT = RUN_DIR / 'p6_diagnostic'
    DIAG_OUT.mkdir(parents=True, exist_ok=True)
    diag_script = PARF_DIR / 'diagnostics' / 'diagnose_v_phi_channels.py'
    result = subprocess.run(
        [
            sys.executable, str(diag_script),
            '--ckpt', str(ckpt_path),
            '--out', str(DIAG_OUT),
            '--n-batches', '4',
            '--batch-size', '8',
            '--block-size', '128',
            '--device', device,
            '--seed', str(SEED),
        ],
        capture_output=True, text=True,
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print('STDERR:\n' + result.stderr[-2000:])
        raise RuntimeError(f'diagnostic exited with code {result.returncode}')

    from IPython.display import Image, display, Markdown
    for fname in ('channels.png', 'gradient_ratio.png'):
        p = DIAG_OUT / fname
        if p.exists():
            display(Markdown(f'### {fname}'))
            display(Image(filename=str(p)))
    summary_md = DIAG_OUT / 'summary.md'
    if summary_md.exists():
        display(Markdown('### diagnostic summary'))
        display(Markdown(summary_md.read_text()))
else:
    print('Skipping V_φ diagnostic.  Set RUN_DIAGNOSTIC = True above to enable.')

In [ ]:
# Per-layer s_ℓ profile (meaningful because --per-layer-v-phi-scale is ON).
if model.raw_v_phi_scale is not None:
    s_ell = F.softplus(model.raw_v_phi_scale).detach().cpu().numpy()
    plt.figure(figsize=(7, 3.5))
    plt.bar(np.arange(1, len(s_ell) + 1), s_ell, color='#3a6ea5')
    plt.axhline(F.softplus(torch.tensor(-3.0)).item(), color='gray',
                linestyle='--', alpha=0.6, label='init s_ℓ ≈ 0.0486')
    plt.xlabel('layer ℓ')
    plt.ylabel('learned s_ℓ = softplus(σ_ℓ)')
    plt.title(f'{CELL} (k={_TOP_K}) per-layer V_φ scale after {STEPS} steps')
    plt.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'per_layer_scale.png', dpi=120)
    plt.show()
    print(f's_ℓ profile = {s_ell.tolist()}')
else:
    print(f'(per_layer_v_phi_scale was OFF for {CELL}.)')

## 9. Save training summary

In [ ]:
summary_path = RUN_DIR / f'{full_tag}_summary.md'
elapsed = time.time() - t0
final_gamma = float(model.gamma.item())
final_tau = model.gumbel_tau if is_sparse else None

with open(summary_path, 'w') as f:
    f.write(f'# Training summary — {full_tag}\n\n')
    f.write(f'- experiment: P10 sparsity ladder at TinyStories scale\n')
    f.write(f'- cell: {CELL}\n')
    f.write(f'- model: SparsePARFLM (full P5+P7+P8 stack)\n')
    f.write(f'- v_phi_kind: {cfg.v_phi_kind}\n')
    f.write(f'- top_k: **{_TOP_K}** (Gumbel-softmax sparse routing)\n')
    f.write(f'- gumbel_tau: {cfg.gumbel_tau_init} -> {cfg.gumbel_tau_min}\n')
    f.write(f'- corpus: TinyStories ({MAX_TRAIN_TOKENS:,} train tokens)\n')
    f.write(f'- params: {n_total:,}  V_theta={n_v_theta:,}  '
            f'V_phi={n_v_phi:,}  score_head={n_score:,}\n')
    f.write(f'- d={cfg.d}  L={cfg.L}  v_hidden={cfg.v_hidden}  '
            f'v_depth={cfg.v_depth}  max_len={cfg.max_len}\n')
    f.write(f'- v_phi inner: phi_hidden={cfg.v_phi_phi_hidden}  '
            f'theta_hidden={cfg.v_phi_theta_hidden}\n')
    f.write(f'- block_size: {train_cfg["block_size"]}  '
            f'batch_size: {train_cfg["batch_size"]}  '
            f'steps: {train_cfg["steps"]}\n')
    f.write(f'- seed: {SEED}\n')
    f.write(f'- elapsed: {elapsed:.0f} s ({elapsed/3600:.2f} h)\n')
    f.write(f'\n## Results\n\n')
    f.write(f'- Best val PPL: **{best_row["val_ppl"]:.2f}** @ step {best_row["step"]}\n')
    f.write(f'- Final val PPL: {log[-1]["val_ppl"]:.2f}\n')
    f.write(f'- Final gamma: {final_gamma:.4f}\n')
    if final_tau is not None:
        f.write(f'- Final gumbel_tau: {final_tau:.4f}\n')
    f.write(f'\n## Comparison to P10g (k=4 baseline)\n\n')
    f.write(f'| Cell | top_k | Best val PPL | Final val PPL | Final γ |\n')
    f.write(f'| --- | ---: | ---: | ---: | ---: |\n')
    f.write(f'| P10g | 4 | 26.42 | 27.16 | 0.134 |\n')
    f.write(f'| {CELL} | {_TOP_K} | {best_row["val_ppl"]:.2f} | '
            f'{log[-1]["val_ppl"]:.2f} | {final_gamma:.3f} |\n')
    delta = best_row['val_ppl'] - 26.42
    f.write(f'\nΔ best PPL vs P10g: {delta:+.2f}\n')
    if delta > 0:
        f.write(f'\n**Verdict:** k={_TOP_K} is worse than k=4 at TinyStories scale '
                f'(+{delta:.2f} PPL). Consistent with the Shakespeare-scale '
                f'sparsity-ladder ordering.\n')
    else:
        f.write(f'\n**Verdict:** k={_TOP_K} BEATS k=4 at TinyStories scale '
                f'({delta:.2f} PPL). SURPRISING — schedule n=3 seed confirmation.\n')

print(f'wrote summary: {summary_path}')

## 10. Sparsity-ladder dashboard

Auto-collects the final and best val PPL from every P10 cell present under `RESULTS_ROOT/{cell}/seed{SEED}/`. Run this notebook twice (once for P10i, once for P10j) and the table below becomes cumulative. Includes the P10g k=4 baseline if its results are on Drive from the earlier P10 notebook runs.

In [ ]:
P10_DIR = RESULTS_ROOT

ALL_CELLS = ('P10a', 'P10b', 'P10c', 'P10d', 'P10e', 'P10f', 'P10g', 'P10h',
             'P10i', 'P10j')
K_MAP = {
    'P10a': 4, 'P10b': 4, 'P10c': 4, 'P10d': 4,
    'P10e': 4, 'P10f': 4, 'P10g': 4, 'P10h': 4,
    'P10i': 8, 'P10j': 16,
}

results = {}
for cell_name in ALL_CELLS:
    cell_dir = P10_DIR / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    if not rows:
        results[cell_name] = None
        continue
    final_ppl = rows[-1]['val_ppl']
    best_ppl = min(r['val_ppl'] for r in rows)
    results[cell_name] = (final_ppl, best_ppl)

P10G_BEST = 26.42
WALL = 33.55

print(f'{"cell":<8} {"k":>4} {"best_ppl":>10} {"final_ppl":>10}    '
      f'{"Δ vs P10g":>10}    verdict')
print('-' * 72)
for cell_name in ALL_CELLS:
    k = K_MAP[cell_name]
    entry = results[cell_name]
    if entry is None:
        print(f'{cell_name:<8} {k:>4} {"—":>10} {"—":>10}    {"—":>10}    (not run)')
        continue
    final_ppl, best_ppl = entry
    delta = best_ppl - P10G_BEST
    if best_ppl <= P10G_BEST:
        verdict = 'beats k=4!'
    elif best_ppl <= WALL:
        verdict = 'beats SPLM wall'
    else:
        verdict = 'above wall'
    print(f'{cell_name:<8} {k:>4} {best_ppl:>10.2f} {final_ppl:>10.2f}    '
          f'{delta:>+10.2f}    {verdict}')

print(f'\nP10g k=4 ceiling = {P10G_BEST}    SPLM em_ln wall = {WALL}    '
      f'MatchedGPT = 7.81')

# Focused sparsity-ladder sub-table
LADDER_CELLS = ('P10g', 'P10i', 'P10j')
ladder_data = [(c, K_MAP[c], results[c]) for c in LADDER_CELLS if results[c] is not None]
if len(ladder_data) >= 2:
    print(f'\n--- Sparsity ladder (TinyStories, d=256, L=8, 16k steps) ---')
    print(f'{"cell":<8} {"k":>4} {"best_ppl":>10} {"final_ppl":>10}    Δ vs k=4')
    print('-' * 56)
    for c, k, entry in ladder_data:
        fp, bp = entry
        delta = bp - P10G_BEST
        print(f'{c:<8} {k:>4} {bp:>10.2f} {fp:>10.2f}    {delta:>+.2f}')

## 11. CLI equivalents (for SLURM / nohup launches)

```bash
# P10i — k=8 sparsity-ladder point (P10g architecture, 16k steps)
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 8 \
    --v-phi-kind structural_competitive \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear \
    --v-phi-phi-hidden 64 --v-phi-theta-hidden 64 \
    --v-hidden 2048
# NOTE: manually set steps=16000 (CLI uses 8000 by default for --mode scaleup)

# P10j — k=16 sparsity-ladder point (P10g architecture, 16k steps)
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 16 \
    --v-phi-kind structural_competitive \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear \
    --v-phi-phi-hidden 64 --v-phi-theta-hidden 64 \
    --v-hidden 2048
# NOTE: manually set steps=16000
```

Output dirs (local CLI): `notebooks/conservative_arch/parf/results/p10_tinystories/{cell}/seed{seed}/`. On Colab: `/content/drive/MyDrive/semsimula_parflm/p10_tinystories/{cell}/seed{seed}/`.